# Deep learning with PyTorch and Lightning

The rest of Module 2 assumes you can read a Lightning training script: it talks about
`LightningDataModule`s, `LightningModule`s and `Trainer`s without stopping to explain them. This
notebook builds that vocabulary on MNIST, then maps every piece onto the file in
`convgru_ensemble/` where IRENE does the same thing with radar.

It is deliberately short. Sections 1–4 cover only the raw PyTorch you need in order to *read* a
Lightning module — tensors, autograd, `nn.Module`, and the `Dataset`/`DataLoader` pair. Sections 5–7
are the Lightning structure itself: **DataModule → LightningModule → Trainer**. There is no
hand-written training loop anywhere: that is exactly the part Lightning takes over.

In [ ]:
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import pytorch_lightning as pl
import torch
import torch.nn.functional as F
from torch import nn
from torch.utils.data import DataLoader, random_split
from torchvision.datasets import MNIST
from torchvision.transforms import Compose, Normalize, ToTensor

pl.seed_everything(42, workers=True)
DATA_DIR = "../data/"

print("torch:", torch.__version__, "| lightning:", pl.__version__)

## 1. Tensors

A `torch.Tensor` is an n-dimensional array with the same semantics as `np.ndarray` — shapes,
slicing, broadcasting, reductions. Two things make it more: it can live on a **GPU**, and it can
**track the operations applied to it** so they can be differentiated (section 2).

In [ ]:
x = torch.tensor([[1.0, 2.0], [3.0, 4.0]])
print("shape:", x.shape, "| dtype:", x.dtype, "| device:", x.device)

# tensor can be moved to GPU if available
x = x.to("cuda" if torch.cuda.is_available() else "cpu")
print("moved to:", x.device)

In [ ]:
# numpy is float64 by default, torch float32
print("numpy default:", np.ones(1).dtype, "| torch default:", torch.ones(1).dtype)

**Note**: Conversion to and from numpy is zero-copy on the CPU: the two objects share memory.

In [ ]:
# they share memory on the CPU
arr = np.ones((2, 2), dtype=np.float32)
t = torch.from_numpy(arr)
arr[0, 0] = 99.0
print("numpy -> torch shares memory:")
print(t)

## 2. Autograd and weights update

Training means nudging parameters in whatever direction reduces the loss, which needs the
**derivative of the loss with respect to every parameter**. 

Tensors created with
`requires_grad=True` record the operations applied to them into a *computational graph*;
`.backward()` walks it backwards applying the chain rule and leaves the result in `.grad`.

You will not call `backward()` yourself again in this notebook — Lightning does it — but this is what
it is doing, and it explains the `.grad`, `zero_grad` and `no_grad` you see in other people's code.

In [ ]:
# a one-layer network, by hand: z = x @ w + b, scored against a target y
x = torch.ones(5)
y = torch.zeros(3)  # yes/no - rain/no rain
w = torch.randn(5, 3, requires_grad=True)
b = torch.randn(3, requires_grad=True)

print("Initial gradients:", w.grad, b.grad)

z = torch.matmul(x, w) + b  # model prdiction
loss = F.binary_cross_entropy_with_logits(z, y)

print("Loss:", loss.item())

loss.backward()
print("New gradients:")
print(w.grad)
print(b.grad)

How do we update the weigths w and bias b? We multiply the gradients by a scalar factor: the learning rate

In [ ]:
learning_rate = 0.1

with torch.no_grad():
    w -= learning_rate * w.grad
    b -= learning_rate * b.grad

    w.grad.zero_()
    b.grad.zero_()

print("Updated weights and biases:")
print(w)
print(b)

Is it our updated model better? Let's check the loss

In [ ]:
z = torch.matmul(x, w) + b  # model prdiction
loss = F.binary_cross_entropy_with_logits(z, y)
print("Loss after update:", loss.item())

**Note**: The learning rate is something we have to decide during training.

## 3. The model: `nn.Module`

A model subclasses `nn.Module`, creates its layers in `__init__` and defines the computation in
`forward`. Anything assigned to `self` is registered, so `.parameters()` finds it — which is how an
optimizer knows what to update.

This is the one PyTorch object Lightning does **not** replace: a `LightningModule` wraps an
`nn.Module`, it does not remove it. Ours is a plain fully-connected net for MNIST digits.

In [ ]:
class SimpleDenseNet(nn.Module):
    """A simple fully-connected neural net for computing predictions."""

    def __init__(self, input_size: int = 784, hidden: int = 256, output_size: int = 10) -> None:
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(input_size, hidden), nn.BatchNorm1d(hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.BatchNorm1d(hidden), nn.ReLU(),
            nn.Linear(hidden, output_size),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # (batch, 1, 28, 28) -> (batch, 784) -> (batch, 10)
        return self.model(x.view(x.size(0), -1))


In [ ]:
# initialize the network
net = SimpleDenseNet()
print(f"parameters: {sum(p.numel() for p in net.parameters()):,}")

# dummy input
inp = torch.randn(2, 1, 28, 28)
plt.imshow(inp[0,0])
plt.show()

# forward pass
out = net(inp)
print("forward:")
print(out)

## 4. Datasets and DataLoaders

A `Dataset` knows two things: how many samples it has (`__len__`) and how to fetch one
(`__getitem__`). `torchvision` ships ready-made ones for common data, MNIST included, so we can try
this out without writing either method ourselves.

In [ ]:
train_ds = MNIST(DATA_DIR, train=True, download=True,
                 transform=Compose([ToTensor(), Normalize((0.1307,), (0.3081,))]))

print("dataset size:", len(train_ds))     # __len__
img, label = train_ds[0]                  # __getitem__
print("one sample:", tuple(img.shape), "-> label", label)

Indexing one sample at a time doesn't batch and doesn't shuffle. A `DataLoader` wraps a `Dataset`
and yields shuffled mini-batches, optionally loaded by background worker processes — this is the
object a training loop actually iterates over.

In [ ]:
train_dl = DataLoader(train_ds, batch_size=128, shuffle=True, num_workers=2)

imgs, labels = next(iter(train_dl))
print("one batch:", tuple(imgs.shape), "->", tuple(labels.shape))

fig, axes = plt.subplots(1, 8, figsize=(10, 1.6))
for ax, im, lab in zip(axes, imgs, labels):
    ax.imshow(im[0], cmap="gray"); ax.set_title(int(lab), fontsize=9); ax.axis("off")
plt.show()

The `LightningDataModule` below wraps exactly this — the same `Dataset` and `DataLoader` — into one
object with a train/val/test split, so the `Trainer` can call it automatically instead of you
passing loaders around by hand.

## 5. Lightning DataModules

A **DataModule** puts every data concern in one place — downloading, splitting, and pass it to the model — so a model can be pointed at a different dataset without touching the model code.

A DataModule implements 5 key methods:

```
def init(self,...):     # runs on 1 process: download, pre-process, save to disk
def setup(self, stage):     # runs on every process: load, split, set state
def train_dataloader(self): # return the train loader
def val_dataloader(self):   # return the validation loader
def test_dataloader(self):  # return the test loader
```

In [ ]:
class MNISTDataModule(pl.LightningDataModule):
    """LightningDataModule for MNIST: download, split, and serve the three loaders."""

    def __init__(
        self,
        data_dir: str = DATA_DIR,
        batch_size: int = 128,
        num_workers: int = 2,
        train_val_split: tuple[int, int] = (55_000, 5_000),
    ) -> None:
        super().__init__()
        # lets us reach the arguments as self.hparams, and stores them in the checkpoint
        self.save_hyperparameters()
        self.transform = Compose([ToTensor(), Normalize((0.1307,), (0.3081,))])
        self.data_train = self.data_val = self.data_test = None

    def setup(self, stage: str | None = None) -> None:
        if self.data_train is None:
            full = MNIST(self.hparams.data_dir, train=True, transform=self.transform)
            self.data_train, self.data_val = random_split(full, self.hparams.train_val_split)
            self.data_test = MNIST(self.hparams.data_dir, train=False, transform=self.transform)

    def _loader(self, dataset, shuffle: bool = False) -> DataLoader:
        return DataLoader(dataset, batch_size=self.hparams.batch_size,
                          num_workers=self.hparams.num_workers, shuffle=shuffle)

    def train_dataloader(self) -> DataLoader:
        return self._loader(self.data_train, shuffle=True)

    def val_dataloader(self) -> DataLoader:
        return self._loader(self.data_val)

    def test_dataloader(self) -> DataLoader:
        return self._loader(self.data_test)


data_module = MNISTDataModule()
data_module.setup()
print("train/val/test:", len(data_module.data_train), len(data_module.data_val), len(data_module.data_test))

imgs, labels = next(iter(data_module.train_dataloader()))
fig, axes = plt.subplots(1, 8, figsize=(10, 1.6))
for ax, img, lab in zip(axes, imgs, labels):
    ax.imshow(img[0], cmap="gray"); ax.set_title(int(lab), fontsize=9); ax.axis("off")
plt.show()

## 6. Lightning Modules

A **LightningModule** holds the model logic. It organises your PyTorch code into 6 sections:

```
- Computations                   (__init__)
- Train loop                     (training_step)
- Validation loop                (validation_step)
- Test loop                      (test_step)
- Prediction loop                (predict_step)
- Optimizers and LR schedulers   (configure_optimizers)
```

Each `*_step` describes what happens for **one batch** and returns the loss. Notice what is absent:
no `.to(device)`, no `loss.backward()`, no `optimizer.step()`, no `zero_grad()`, no `train()`/`eval()`
toggling, no epoch loop. The `Trainer` does all of it.

In [ ]:
class MNISTLitModule(pl.LightningModule):
    """LightningModule for MNIST classification with a dense network."""

    def __init__(self, hidden: int = 256, lr: float = 1e-4) -> None:
        super().__init__()
        self.save_hyperparameters()
        self.net = SimpleDenseNet(hidden=hidden)
        self.criterion = nn.CrossEntropyLoss()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)

    def model_step(self, batch: Any) -> tuple[torch.Tensor, torch.Tensor]:
        x, y = batch
        logits = self(x)
        loss = self.criterion(logits, y)
        return loss

    def _shared_step(self, batch: Any, split: str) -> torch.Tensor:
        loss = self.model_step(batch)
        self.log(f"{split}_loss", loss, prog_bar=True)
        return loss

    def training_step(self, batch: Any, batch_idx: int) -> torch.Tensor:
        return self._shared_step(batch, "train")

    def validation_step(self, batch: Any, batch_idx: int) -> torch.Tensor:
        return self._shared_step(batch, "val")

    def test_step(self, batch: Any, batch_idx: int) -> torch.Tensor:
        return self._shared_step(batch, "test")

    def configure_optimizers(self) -> dict:
        optimizer = torch.optim.Adam(self.parameters(), lr=self.hparams.lr)
        return {"optimizer": optimizer}

model = MNISTLitModule()
print(model)

## 7. Lightning Trainer

The **Trainer** owns the loops. It picks the accelerator, moves the batches, runs validation every
epoch, and runs the test set when you ask. `fit` takes the module and the datamodule; nothing else
needs wiring.

In [ ]:
trainer = pl.Trainer(max_epochs=1)

trainer.fit(model, datamodule=data_module)      # train and validate
trainer.test(model, datamodule=data_module)     # test

### Checking the predictions

The metrics above are averages over the whole test set — the qualitative check is to actually look
at a handful of predictions. Pull a random batch straight from the test set (not through the
`DataLoader` used for training, so the order is different every run) and compare what the model
says to the true label.

In [ ]:
model.eval()
sample_loader = DataLoader(data_module.data_test, batch_size=12, shuffle=True)
imgs, labels = next(iter(sample_loader))

with torch.no_grad():
    outputs = model(imgs)

print(outputs.shape)

In [ ]:
preds = outputs.argmax(dim=1)

fig, axes = plt.subplots(2, 6, figsize=(11, 4))
for ax, img, pred, true in zip(axes.flat, imgs, preds, labels):
    correct = pred == true
    ax.imshow(img[0], cmap="gray")
    ax.set_title(f"pred {pred.item()}")
    ax.axis("off")
plt.tight_layout()
plt.show()

## 8. Where each piece shows up in IRENE

Everything above has a counterpart in `convgru_ensemble/`. When you open those files later, this is
the map:

| Here | In IRENE |
|---|---|
| `SimpleDenseNet(nn.Module)` | `EncoderDecoder` — ConvGRU encoder–decoder, noise fed to the decoder to make an ensemble |
| `MNIST(...)` / `DataLoader(...)` | `SampledRadarDataset` — one item is a `(T, H, W)` radar datacube, not an image |
| `MNISTDataModule` | `RadarDataModule` — same five methods, over the Zarr archive |
| `MNISTLitModule` | `RadarLightningModel` — its `shared_step` is the one you just wrote |
| `nn.CrossEntropyLoss()` | CRPS — scores a whole ensemble, not a single prediction |
| `trainer.fit(...)` | `convgru_ensemble.train`, launched from the CLI with a config file |
| `(batch, 1, 28, 28)` | `(B, T, C, H, W)` — time is its own axis, channels carry ensemble members |

**Shapes are the thing to watch.** Our model flattens the image because a dense layer takes vectors;
IRENE keeps `(B, T, C, H, W)` because its ConvGRU cells step through time explicitly. Most errors you
hit when modifying a model are shape errors, and the message tells you which axis disagreed.

## Recap

- A **tensor** is an array that also knows its device and its gradient.
- **Autograd** records operations and differentiates them on `.backward()`.
- A model is an **`nn.Module`**; Lightning wraps one, it does not replace it.
- A **`Dataset`** answers `len()` and indexing; a **`DataLoader`** turns it into shuffled mini-batches.
- A **DataModule** owns the data (5 methods), a **LightningModule** owns the logic (6 sections).
- The **Trainer** owns the loops — always spot-check predictions on a sample, not just the averaged
  metrics.

Next: **`importance_sampling_demo.ipynb`** and **`training_pipeline_demo.ipynb`** — the same
structure, with real radar datacubes out of the ArcoDataHub archive.